# Ablation Study: Component Analysis of TE-Q-Transformer
## Controlled Empirical Experiments

**Research Objective:** Systematically evaluate and decouple the architectural components of the TE-Q-Transformer framework:
1. **Physics-Guided Arrhenius Temperature Encoding vs. Conventional Temperature Scaling** (E02)
2. **Degradation Kinetics Mechanisms: SEI-only vs. Plating-only vs. Dual-Mechanism** (E03)
3. **Variational Quantum Feature Map vs. Parameter-Matched Classical Representation** (E04)

---

This notebook provides the implementation code to instantiate each ablation variant, train under identical conditions, and compare validation trajectories.

In [ ]:
# ==============================================================================
# 0. SETUP ENVIRONMENT AND REPOSITORY PATHS
# ==============================================================================
import os
import sys
from pathlib import Path

MANUAL_REPO_ROOT = None
REPO_NAME = "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation"

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/kaggle/working") / REPO_NAME,
    Path("/kaggle/working"),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

if MANUAL_REPO_ROOT and Path(MANUAL_REPO_ROOT).exists():
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    REPO_ROOT = next(
        (c.resolve() for c in CANDIDATES if (c / "models" / "proposed" / "te_q_transformer.py").exists() or (c / "datasets" / "NASA" / "processed").exists()),
        Path.cwd().resolve()
    )

print(f"[Setup] REPO_ROOT resolved to: {REPO_ROOT}")
DATA_ROOT = REPO_ROOT / "datasets"
MODEL_ROOT = REPO_ROOT / "models"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml

import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Hardware device: {DEVICE}")

## 1. Load Dataset
Loading the multi-temperature NASA Ames dataset partitioned according to the non-leakage protocol.

In [ ]:
from src.data.nasa_loader import get_nasa_dataloaders

nasa_dir = DATA_ROOT / "NASA" / "processed"
train_loader, test_loaders, scaler = get_nasa_dataloaders(nasa_dir, batch_size=8)
print(f"[Dataset] Train: {len(train_loader.dataset)} cycles across {len(train_loader)} batches")
for cell_id, loader in test_loaders.items():
    print(f"  - Test cell {cell_id:<12}: {len(loader.dataset)} cycles")

## 2. Defining Ablation Architectures
Defining modular variants for:
1. **Full TE-Q-Transformer**: Dual Arrhenius kinetics (SEI + Plating) + 4-qubit VQC.
2. **Classical MLP Counterpart (E04)**: Replacing the 4-qubit quantum circuit with a parameter-matched classical feed-forward module.
3. **Standard Linear Transformer (E02)**: Direct linear projection of raw numerical features without physical embedding.

In [ ]:
from models.proposed import TEQTransformer, rich_entangler_config
from models.baselines.transformer import TransformerModel

class ClassicalMLPEmbedding(nn.Module):
    """Parameter-matched classical replacement for the 4-qubit VQC layer."""
    def __init__(self, input_dim: int = 4, output_dim: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.Tanh(),
            nn.Linear(16, output_dim),
            nn.Tanh()
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4]
        return self.net(x)

class ClassicalAblationModel(nn.Module):
    """TE-Q-Transformer architecture with quantum circuit replaced by Classical MLP."""
    def __init__(self):
        super().__init__()
        self.classical_embed = ClassicalMLPEmbedding(4, 4)
        self.input_proj = nn.Linear(4, 64)
        self.temporal_smooth = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64, nhead=2, dim_feedforward=64, dropout=0.0, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.head = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.classical_embed(x)
        h = self.input_proj(feat)
        h = self.temporal_smooth(h.transpose(1, 2)).transpose(1, 2)
        h = self.transformer(h)
        out = self.head(h.mean(dim=1))
        return out.squeeze(-1)

ablation_models = {
    "TE-Q-Transformer (Full Physics + Quantum)": TEQTransformer(rich_entangler_config()),
    "Classical MLP Variant (E04)": ClassicalAblationModel(),
    "Raw Features Classical Transformer (E02)": TransformerModel(),
}

print("=" * 75)
print(f"{'Ablation Variant':<45} | {'Trainable Parameters':<20}")
print("=" * 75)
for name, m in ablation_models.items():
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:<45} | {n_params:<20,d}")
print("=" * 75)

## 3. Training & Evaluation Engine
Modular pipeline to run controlled training epochs and assess validation performance.

In [ ]:
from src.eval.metrics import calculate_metrics, calculate_macro_metrics

def train_epoch(model: nn.Module, loader: DataLoader, optimizer: optim.Optimizer, criterion: nn.Module, device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        pred = model(bx)
        loss = criterion(pred, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(by)
    return total_loss / len(loader.dataset)

def evaluate_model(model: nn.Module, test_loaders: dict, device: torch.device) -> dict:
    model.eval()
    per_cell = {}
    with torch.no_grad():
        for cell_id, loader in test_loaders.items():
            preds, actuals = [], []
            for bx, by in loader:
                bx = bx.to(device)
                pred = model(bx)
                preds.append(pred.cpu().numpy())
                actuals.append(by.numpy())
            y_pred = np.concatenate(preds)
            y_true = np.concatenate(actuals)
            per_cell[cell_id] = {
                "metrics": calculate_metrics(y_true, y_pred),
                "preds": y_pred,
                "actuals": y_true,
            }
    macro = calculate_macro_metrics([res["metrics"] for res in per_cell.values()])
    return {"per_cell": per_cell, "macro": macro}

def run_ablation_experiment(model: nn.Module, model_name: str, epochs: int = 5, lr: float = 1e-3, device: torch.device = DEVICE) -> dict:
    print(f"\n>>> Running Ablation: {model_name} <<<")
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    for ep in range(1, epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, criterion, device)
        print(f"Epoch {ep:2d}/{epochs:2d} | Train Loss: {loss:.6f}")
    
    eval_res = evaluate_model(model, test_loaders, device)
    macro = eval_res["macro"]
    print(f"--> Macro RMSE: {macro['RMSE']:.5f} | MAE: {macro['MAE']:.5f} | R2: {macro['R2']:.5f}")
    return eval_res

## 4. Run Example Ablation Comparison
Execute training on the classical MLP counterpart vs the raw classical transformer.

In [ ]:
# Example: Train and evaluate Classical MLP Variant
res_classical = run_ablation_experiment(
    ablation_models["Classical MLP Variant (E04)"],
    "Classical MLP Variant (E04)",
    epochs=5,
    lr=1e-3,
    device=DEVICE
)